# Data Loading and Preparation

## Basic Imports

In [ ]:
# Data handling
import pandas as pd
import numpy as np
from collections import Counter

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Pathing
from pathlib import Path

# Arithmetic
from math import sqrt

# Statistics
from scipy.stats import skew

# PCA
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Make plots look nicer
sns.set_theme(style="whitegrid")
%matplotlib inline

## Load Flagged Flow Dataset for Analysis

In [ ]:
flagged_flows_path = Path("../data/processed/stage1_flagged_flows.csv")
flagged_flows = pd.read_csv(flagged_flows_path, low_memory=False)

# Sanity check everything flagged
print(flagged_flows["pred_anomaly"].value_counts())
print(flagged_flows["outcome"].value_counts())

# Show first few rows
display(flagged_flows.head())

# Check shape
print("Dataset shape:", flagged_flows.shape)

## Visualise Traffic Class Balance After Filtering

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(x="Label", data=flagged_flows)
plt.xlabel("Traffic Type (0 = Normal, 1 = Attack)")
plt.ylabel("Count")
plt.title("Normal vs Attack Traffic")
plt.show()

# Numeric Features

## Candidate Features

In [ ]:
# Temporal decomposition
Sdt = pd.to_datetime(flagged_flows["Stime"], unit="s")
flagged_flows["Shourofday"] = Sdt.dt.hour
flagged_flows["Sdayofweek"] = Sdt.dt.dayofweek

Ddt = pd.to_datetime(flagged_flows["Ltime"], unit="s")
flagged_flows["Lhourofday"] = Ddt.dt.hour
flagged_flows["Ldayofweek"] = Ddt.dt.dayofweek

numeric_features = flagged_flows.select_dtypes(include=np.number).columns.tolist()
numeric_features = [feat for feat in numeric_features if feat not in ["Label", "pred_anomaly"]] # Exclude targets
print("Number of numeric features:", len(numeric_features))

## Compute Comprehensive Feature Summaries

In [ ]:
# Boolean masks for TP/FP
tp_mask = flagged_flows["outcome"] == "TP"
fp_mask = flagged_flows["outcome"] == "FP"

# Slice numeric features
tp_df = flagged_flows.loc[tp_mask, numeric_features]
fp_df = flagged_flows.loc[fp_mask, numeric_features]

# Cohen's d
mean_diff = tp_df.mean() - fp_df.mean()
pooled_std = np.sqrt((tp_df.std()**2 + fp_df.std()**2)/2)
cohen_d = mean_diff/pooled_std

# IQR
tp_iqr = tp_df.quantile(0.75) - tp_df.quantile(0.25)
fp_iqr = fp_df.quantile(0.75) - fp_df.quantile(0.25)

# Tails
tp_95, fp_95 = tp_df.quantile(0.95), fp_df.quantile(0.95)
tp_5, fp_5 = tp_df.quantile(0.05), fp_df.quantile(0.05)

# Skew
tp_skew = skew(tp_df, axis=0, bias=False)
fp_skew = skew(fp_df, axis=0, bias=False)

# Tail differences
tail_diff_95_scaled = (tp_95 - fp_95)/(tp_iqr + fp_iqr)
tail_diff_95_scaled = tail_diff_95_scaled.replace([np.inf, -np.inf], np.nan)
tail_diff_5_scaled = (tp_5 - fp_5)/(tp_iqr + fp_iqr)
tail_diff_5_scaled = tail_diff_5_scaled.replace([np.inf, -np.inf], np.nan)

# Skew difference
skew_diff = tp_skew - fp_skew

# Build summary DataFrame
feature_summary = pd.DataFrame({
    "Feature": numeric_features,
    "Cohen_d": cohen_d.values,
    "TP_median": tp_df.median(),
    "FP_median": fp_df.median(),
    "TP_IQR": tp_iqr.values,
    "FP_IQR": fp_iqr.values,
    "TP_95th": tp_95.values,
    "FP_95th": fp_95.values,
    "TP_5th": tp_5.values,
    "FP_5th": fp_5.values,
    "TP_skew": tp_skew,
    "FP_skew": fp_skew,
    "Tail_diff_95_scaled": tail_diff_95_scaled.values,
    "Tail_diff_5_scaled": tail_diff_5_scaled.values,
    "Skew_diff": skew_diff
})

# Screening lens
sorted_feature_summary = feature_summary.sort_values("Cohen_d", key=abs, ascending=False)
display(sorted_feature_summary)

## Visualise Top Feature Distributions

### Cohen's D (Mean Shift)

In [ ]:
sorted_by_cohen = feature_summary.sort_values("Cohen_d", key=abs, ascending=False)
top_cohen_features = sorted_by_cohen.head(7)["Feature"].tolist()
for feat in top_cohen_features:
    plt.figure(figsize=(6, 4))

    # Violin layer
    sns.violinplot(
        x="outcome",
        y=feat,
        data=flagged_flows,
        palette={"TP": "red", "FP": "orange"},
        hue="outcome",
        inner=None,
        alpha=0.5,
        dodge=False
    )

    # Box layer
    sns.boxplot(
        x="outcome",
        y=feat,
        data=flagged_flows,
        width=0.2,
        palette={"TP": "red", "FP": "orange"},
        hue="outcome",
        legend=False,
        dodge=False
    )

    plt.xlabel("Outcome")
    plt.ylabel(feat)
    plt.title(f"Distribution of {feat} by Outcome")
    plt.show()

### Combine Candidates

In [ ]:
abs_metrics = feature_summary[["Cohen_d", "Tail_diff_95_scaled", "Tail_diff_5_scaled", "Skew_diff"]].abs()

# Normalise each column to [0, 1] range
abs_metrics_norm = (abs_metrics - abs_metrics.min())/(abs_metrics.max() - abs_metrics.min())

# Composite score: sum of normalised metrics
feature_summary["Composite_score"] = abs_metrics_norm.sum(axis=1)

# Rank features
feature_summary = feature_summary.sort_values("Composite_score", ascending=False)
display(feature_summary[["Feature", "Composite_score"]])

top_features = feature_summary.head(15)["Feature"].tolist()
print("Top features:", top_features)

### Layered Plots for Combined Candidate Set

In [ ]:
plt.figure(figsize=(6, 4))

# Violin layer
sns.violinplot(
    x="outcome",
    y="dwin",
    data=flagged_flows,
    palette={"TP": "red", "FP": "orange"},
    hue="outcome",
    inner=None,
    alpha=0.5,
    dodge=False
)

# Box layer
sns.boxplot(
    x="outcome",
    y="dwin",
    data=flagged_flows,
    width=0.2,
    palette={"TP": "red", "FP": "orange"},
    hue="outcome",
    legend=False,
    dodge=False
)

plt.xlabel("Outcome")
plt.ylabel("dwin")

In [ ]:
for feat in top_features:
    plt.figure(figsize=(6, 4))

    # Violin layer
    sns.violinplot(
        x="outcome",
        y=feat,
        data=flagged_flows,
        palette={"TP": "red", "FP": "orange"},
        hue="outcome",
        inner=None,
        alpha=0.5,
        dodge=False
    )

    # Box layer
    sns.boxplot(
        x="outcome",
        y=feat,
        data=flagged_flows,
        width=0.2,
        palette={"TP": "red", "FP": "orange"},
        hue="outcome",
        legend=False,
        dodge=False
    )

    plt.xlabel("Outcome")
    plt.ylabel(feat)
    plt.title(f"Distribution of {feat} by Outcome")
    plt.show()

## Plot Paired Feature Interactions

### Cohen's D (Mean Shift)

In [ ]:
sns.pairplot(
    flagged_flows,
    vars=top_cohen_features,
    hue="outcome",
    palette={"TP": "red", "FP": "orange"},
    diag_kind="kde",
    corner=True
)
plt.suptitle("Top Feature Interactions by Outcome", y=1.02)
plt.show()

### Combined Candidate Feature Set

#### Pair Plots

In [ ]:
sns.pairplot(
    flagged_flows,
    vars=top_features,
    hue="outcome",
    palette={"TP": "red", "FP": "orange"},
    diag_kind="kde",
    corner=True,
    markers=["o", "s"],
    height=2.5
)
plt.suptitle("Top Feature Interactions by Outcome", y=1.02)
plt.show()

#### Correlation Matrix

In [ ]:
corr_matrix = flagged_flows[top_features].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool)) # Hide upper triangle (keep lower visible)
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", )
plt.title("Correlation Matrix for Top Features")
plt.show()

`dloss`, `Dpkts`, and `dbytes` are perfectly correlated across the flagged flows. Including all three in the model would be redundant; keeping only a single representative captures the signal while improving interpretability. The same holds for `sloss`, `Spkts`, and `sbytes`, which also form a perfectly correlated trio.

## Test Interaction Feature Effects

### `sttl` and `ttl_diff`

#### Ratio Test

Tested whether collapsing `sttl` and `ttl_diff` into a ratio produced univariate separation between TP and FP flows. 

Result: Low Cohen's d and identical medians, indicating no meaningful marginal separation.

Conclusion: Separation appears to be interaction-based rather than reducible to a single scalar feature.

In [ ]:
sttl_over_ttl_diff = flagged_flows["sttl"]/(flagged_flows["ttl_diff"] + 1e-6)
tp_vals = sttl_over_ttl_diff[flagged_flows["outcome"] == "TP"]
fp_vals = sttl_over_ttl_diff[flagged_flows["outcome"] == "FP"]
mean_diff = tp_vals.mean() - fp_vals.mean()
pooled_std = np.sqrt((tp_vals.std()**2 + fp_vals.std()**2)/2)
cohen_d = mean_diff/pooled_std if pooled_std != 0 else 0
interaction_summary = pd.DataFrame({
    "Feature": ["sttl_over_ttl_diff"],
    "Cohen_d": [cohen_d],
    "TP_median": [tp_vals.median()],
    "FP_median": [fp_vals.median()],
    "TP_IQR": [tp_vals.quantile(0.75) - tp_vals.quantile(0.25)],
    "FP_IQR": [fp_vals.quantile(0.75) - fp_vals.quantile(0.25)]
})
display(interaction_summary)

#### Product Test

Tested whether collapsing `sttl` and `ttl_diff` into a product produced univariate separation between TP and FP flows.

Result: Higher Cohen's d but still slightly weaker than `ttl_diff` alone.

Conclusion: The raw product doesn't capture any additional separation.

In [ ]:
sttl_x_ttl_diff = flagged_flows["sttl"]*flagged_flows["ttl_diff"]
tp_vals = sttl_x_ttl_diff[flagged_flows["outcome"] == "TP"]
fp_vals = sttl_x_ttl_diff[flagged_flows["outcome"] == "FP"]
mean_diff = tp_vals.mean() - fp_vals.mean()
pooled_std = np.sqrt((tp_vals.std()**2 + fp_vals.std()**2)/2)
cohen_d = mean_diff/pooled_std if pooled_std != 0 else 0
interaction_summary = pd.DataFrame({
    "Feature": ["sttl_x_ttl_diff"],
    "Cohen_d": [cohen_d],
    "TP_median": [tp_vals.median()],
    "FP_median": [fp_vals.median()],
    "TP_IQR": [tp_vals.quantile(0.75) - tp_vals.quantile(0.25)],
    "FP_IQR": [fp_vals.quantile(0.75) - fp_vals.quantile(0.25)]
})
display(interaction_summary)

Neither ratio nor product of `sttl` and `ttl_diff` produced stronger univariate separation than `ttl_diff` alone, suggesting that TP/FP separation is inherently 2-dimensional.

### `sttl` and `ct_state_ttl`

#### Product Test

In [ ]:
sttl_x_ct_state_ttl = flagged_flows["sttl"]*flagged_flows["ct_state_ttl"]
tp_vals = sttl_x_ct_state_ttl[flagged_flows["outcome"] == "TP"]
fp_vals = sttl_x_ct_state_ttl[flagged_flows["outcome"] == "FP"]

# Cohen's d
mean_diff = tp_vals.mean() - fp_vals.mean()
pooled_std = np.sqrt((tp_vals.std()**2 + fp_vals.std()**2)/2)
cohen_d = mean_diff/pooled_std

# IQR
tp_iqr = tp_vals.quantile(0.75) - tp_vals.quantile(0.25)
fp_iqr = fp_vals.quantile(0.75) - fp_vals.quantile(0.25)

# Tails
tp_95, fp_95 = tp_vals.quantile(0.95), fp_vals.quantile(0.95)
tp_5, fp_5 = tp_vals.quantile(0.05), fp_vals.quantile(0.05)

# Skew
tp_skew = skew(tp_vals, bias=False)
fp_skew = skew(fp_vals, bias=False)

# Tail differences
tail_diff_95_scaled = (tp_95 - fp_95)/(tp_iqr + fp_iqr + 1e-6)
tail_diff_5_scaled = (tp_5 - fp_5)/(tp_iqr + fp_iqr + 1e-6)

# Skew difference
skew_diff = tp_skew - fp_skew

interaction_summary = pd.DataFrame({
    "Feature": ["sttl_x_ct_state_ttl"],
    "Cohen_d": [cohen_d],
    "TP_median": tp_vals.median(),
    "FP_median": fp_vals.median(),
    "TP_IQR": tp_iqr,
    "FP_IQR": fp_iqr,
    "TP_95th": tp_95,
    "FP_95th": fp_95,
    "TP_5th": tp_5,
    "FP_5th": fp_5,
    "TP_skew": tp_skew,
    "FP_skew": fp_skew,
    "Tail_diff_95_scaled": tail_diff_95_scaled,
    "Tail_diff_5_scaled": tail_diff_5_scaled,
    "Skew_diff": skew_diff
})
display(interaction_summary)

feature_summary_combined = pd.concat([feature_summary, interaction_summary], ignore_index=True)
abs_metrics = feature_summary_combined[["Cohen_d", "Tail_diff_95_scaled", "Tail_diff_5_scaled", "Skew_diff"]].abs()
abs_metrics_norm = (abs_metrics - abs_metrics.min())/(abs_metrics.max() - abs_metrics.min())
feature_summary_combined["Composite_score"] = abs_metrics_norm.sum(axis=1)
feature_summary_combined = feature_summary_combined.sort_values("Composite_score", ascending=False)
display(feature_summary_combined[["Feature", "Composite_score"]])

## Revisualise Using Other Plots

### Density Plots

#### STTL vs TTL Diff

In [ ]:
plt.figure(figsize=(8, 6))
sns.kdeplot(
    data=flagged_flows,
    x="sttl",
    y="ttl_diff",
    hue="outcome",
    palette={"TP": "red", "FP": "orange"},
    common_norm=False,
    alpha=0.8,
)
plt.xlabel("STTL")
plt.ylabel("TTL Diff")
plt.title("2D Density of STTL vs TTL Diff")
plt.show()

#### STTL vs Duration

In [ ]:
plt.figure(figsize=(8, 6))
sns.kdeplot(
    data=flagged_flows,
    x="sttl",
    y="dur",
    hue="outcome",
    palette={"TP": "red", "FP": "orange"},
    common_norm=False,
    alpha=0.8
)
plt.xlabel("STTL")
plt.ylabel("Duration")
plt.title("2D Density of STTL vs Duration")
plt.show()

## PCA Analysis

### Prepare Matrix

In [ ]:
X = flagged_flows[numeric_features].copy()
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Shape of scaled matrix:", X_scaled.shape)

### Fit PCA

In [ ]:
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

### Visualise PCA

In [ ]:
explained_variance = pca.explained_variance_ratio_
plt.figure(figsize=(8, 6))
plt.plot(np.cumsum(explained_variance))
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA Explained Variance")
plt.show()

Takes ~8 components to explain 80% of variance:
- Feature space is not low-dimensional
- Information spread across many directions
- No single axis dominating structure

In [ ]:
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)
pca_df = pd.DataFrame({
    "PC1": X_pca_2d[:, 0],
    "PC2": X_pca_2d[:, 1],
    "outcome": flagged_flows.loc[X.index, "outcome"].values
})

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="outcome",
    palette={"TP": "red", "FP": "orange"},
    alpha=0.6
)
plt.title("PCA Projection (2D)")
plt.show()

Heavy overlap implies directions of maximum variance are not directions of maximum class discrimination. TP vs FP differences are either:
- Nonlinear
- Weak relative to overall variance
- Spread across many components

# Categorical Features

## Initial Exploration of Distributions

### Identify Categorical Features

In [ ]:
categorical_features = flagged_flows.select_dtypes(include=["category", "object", "str"]).columns.tolist()
low_card_numeric = [col for col in flagged_flows.select_dtypes(include=[int, float]).columns if flagged_flows[col].nunique() <= 10]
categorical_features += low_card_numeric
print(categorical_features)

### Inspect Select Features

In [ ]:
for col in ["sport", "dsport", "state", "service", "proto"]:
    print(f"\nValue counts for {col}:")
    print(flagged_flows[col].value_counts().head(10).sort_values(ascending=False)) # Top 10 categories

### Examine Key Distributions

In [ ]:
tp_subset = flagged_flows[flagged_flows["Label"] == 1]
fp_subset = flagged_flows[flagged_flows["Label"] == 0]
for col in ["state", "service", "proto", "dsport"]:
    tp_counts = tp_subset[col].value_counts()
    tp_frac = tp_counts/tp_counts.sum()
    tp_df = pd.DataFrame({
        "tp_counts": tp_counts,
        "tp_frac": tp_frac
    }).sort_values(by="tp_counts", ascending=False)
    print(f"\nTop {col} values for true positives:")
    display(tp_df)

    fp_counts = fp_subset[col].value_counts()
    fp_frac = fp_counts/fp_counts.sum()
    fp_df = pd.DataFrame({
        "fp_counts": fp_counts,
        "fp_frac": fp_frac
    }).sort_values(by="fp_counts", ascending=False)
    print(f"\nTop {col} values for false positives:")
    display(fp_df)

    df_frac = pd.DataFrame({
        "TP": tp_frac,
        "FP": fp_frac
    }).fillna(0).sort_values(by="TP", ascending=False)
    df_frac.head(10).plot(kind="bar", figsize=(10, 6), color={"TP": "red", "FP": "blue"})
    plt.title(f"{col.capitalize()} Distribution: TP vs FP Fractions")
    plt.xlabel(col.capitalize())
    plt.ylabel("Fraction of Flows")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## Thorough Examination of Candidates

### Distribution of `is_smips_ports` Across Outcomes

In [ ]:
tp_subset = flagged_flows[flagged_flows["Label"] == 1]
fp_subset = flagged_flows[flagged_flows["Label"] == 0]

tp_counts = tp_subset["is_sm_ips_ports"].value_counts().sort_index()
fp_counts = fp_subset["is_sm_ips_ports"].value_counts().sort_index()
print(tp_counts, fp_counts)

df_counts = pd.DataFrame({
    "FP": fp_counts,
    "TP": tp_counts
}).fillna(0)

# Plot as bar chart
df_counts.plot(kind="bar", figsize=(8, 6), color={"FP": "blue", "TP": "red"})
plt.title("Distribution of is_sm_ips_ports by Outcome")
plt.xlabel("is_sm_ips_ports")
plt.ylabel("Number of flows")
plt.xticks([0, 1], ["0 (unique)", "1 (repeated)"], rotation=0)
plt.show()

All true positives have unique IP-port combinations while a few false positives are repeated. Although this feature could provide strong discriminatory power in the model, such a distribution is unlikely to generalise to real-world traffic, where attacks may also repeat source-destination IP-port combinations.

### Distribution of Transaction Depth Across Outcomes

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(
    data=flagged_flows,
    x="trans_depth",
    hue="outcome",
    palette={"FP": "blue", "TP": "red"}
)
plt.title("Distribution of Transaction Depth by Outcome")
plt.xlabel("Transaction Depth")
plt.ylabel("Count")
plt.show()

display(
    flagged_flows[flagged_flows["Label"] == 0]["trans_depth"]
    .value_counts()
    .sort_index()
)

display(
    flagged_flows[flagged_flows["Label"] == 1]["trans_depth"]
    .value_counts()
    .sort_index()
)